In [1]:
from collections import deque
import heapq
import math 

In [2]:
class node_trie: 
    def __init__ (self): 
        self.fin= False 
        self.hijos= {}
        self.palabras_dif=0
        
class Trie: 
    
    def __init__(self): 
        self.raiz= node_trie()
        self.cont= 0
        self.palabras_dif=0
        
            
    def inserta (self, palabra):
        nodo= self.raiz
        for letra in palabra:
            if letra not in nodo.hijos:
                nodo.hijos[letra]= node_trie()

            nodo= nodo.hijos[letra]

        nodo.fin= True

    def buscaRec(self, palabra, actual=node_trie()):
        if actual is None:
            return False
        if len(palabra)==0:
            return actual.fin
        if palabra[0] not in actual.hijos:
            return False
        else:
            return self.buscaRec(palabra[1:], actual.hijos[palabra[0]])
    
    def busca(self, palabra):
         print(self.buscaRec(palabra=palabra, actual=self.raiz))

    def autocompletado(self, prefijo):
        nodo = self.raiz

        for letra in prefijo:
            if letra not in nodo.hijos:
                print("No palabras con su prefijo")
                return
            nodo = nodo.hijos[letra]
            
        self.autocompletadoR(nodo, prefijo)

    def autocompletadoR(self, nodo, palabra_actual):
        if nodo.fin:
            print(palabra_actual)

        for letra, siguiente in nodo.hijos.items():
            self.autocompletadoR(siguiente, palabra_actual + letra)

In [3]:
class grafos:
    def __init__ (self):
        self.G = None
        self.hijos= {}
        
    def __DFS(self, actual, lista, visitado):
        if actual is None:
            return
        if visitado[actual]:
            return
        
        visitado[actual] = True
        for hijo in G[actual]:
            if not visitado[hijo]:
                lista.append(hijo)
                self.__DFS(hijo, lista, visitado)
    
    def DFS(self):
        visitado = {}
        for v in G:
            visitado[v] = False
        lista = []
        for v in self.G:
            if not visitado[v]:
                self.__DFS(v, lista, visitado)
        return lista
                
    
    def BFS(self,lista,inicio):
        visitado = {}
        for v in G:
            visitado[v] = False
            
        lista=[]
        cola= deque()
        cola= cola.append(inicio)
        
        while len(cola)>0:
            actual=cola.popleft()
            
            if not visitado[actual]:
                visitado[actual] = True
                lista.append(actual)
                
            for hijo in self.G[actual]:
                if not visitado[hijo]:
                    cola.append(hijo)
        return lista
    
    def prim(self, inicio):                                                                               
        pi = {}                                                                                           
        llave = {}                                                                                        
                                                                                                        
        for v in self.G:                                                                                  
            pi[v] = None                                                                                  
            llave[v] = float('inf')                                                                       
        llave[inicio] = 0                                                                                 
                                                                                                        
        heap = []                                                                                         
        heapq.heappush(heap, (0, inicio))
        # Este es un conjunto de nodos que ya se han visitado, para no volver a visitarlos y evitar ciclos.                                                                 
        visitado = set()                                                                                  
                                                                                                        
        while len(heap) > 0:                                                                              
            peso, v = heapq.heappop(heap)

            #Si ya se visitó el nodo, se salta el resto del while                                                                                             
            if v in visitado:                                                                             
                continue                                                                                  
            visitado.add(v)                                                                               
                                                                                                        
            for k in self.G[v]:                                                                           
                if k not in visitado and self.G[v][k] < llave[k]:                                         
                    llave[k] = self.G[v][k]                                                               
                    pi[k] = v                                                                             
                    heapq.heappush(heap, (llave[k], k))                                                   
        return pi         
        
    def dijkstra(self, inicio):
        pi = {}
        llave = {}
        
        for v in self.G:
            pi[v] = None
            llave[v] = float('inf')
        
        llave[inicio] = 0
        heap = []
        
        heapq.heappush(heap, (0, inicio))
        
        while len(heap) > 0:
            distancia_actual, u = heapq.heappop(heap)
            
            for v in self.G[u]:
                
                if llave[u] + self.G[u][v] < llave[v]:
                    #Llave v va a meter la distancia mínima acumulada al punto de inicio 
                    llave[v] = llave[u] + self.G[u][v]
                    #Va a meter por dónde llegó 
                    pi[v] = u
                    heapq.heappush(heap, (llave[v], v))
        #El valor de llave para cada punto es la distancia mínima del inicio a ese punto 
        #El valor de pi, te va a decir el último que usó para llegar a ese nodo            
        return llave, pi


In [4]:
class Nodo_arbol: 
    def __init__ (self, punto): 
        self.punto= punto 
        self.izq= None 
        self.der= None
        
class kd_trees:
    def __init__(self, k):
        self.h = k
        self.raiz= None 
        
    def construir_arbol(self, puntos, profundidad):
        if not puntos:
            return None

       # Queremos ver contra qué eje lo vamos a comparar
        eje = profundidad % self.h
        #Ordenamos conforme a la mediana conforme a un eje para poder insertarlo
        puntos.sort(key = lambda punto : punto[eje]) 
        mediana = len(puntos) // 2
        nodo = Nodo_arbol(punto=puntos[mediana])

        if profundidad==0: 
            self.raiz= nodo
            
        nodo.izq = self.construir_arbol(puntos[:mediana], profundidad + 1)
        nodo.der = self.construir_arbol(puntos[mediana + 1:], profundidad + 1)

        return nodo
        
    def busca (self, actual, target, nivel): 
        if actual is None:
            return None
            
        if self.distancia(actual.punto, target)==0: 
            return actual
        
        c= nivel% self.h
        if target[c]<= actual.punto[c]: 
            return self.busca(actual.izq, target, nivel+1)
        else: 
            return self.busca(actual.der, target, nivel+1)
        
    def buscar_min(self, actual, target, nivel, mejorp, mejord): 
        #nota: target es el que le meto, actual es sobre el que estoy iterando 
        if actual is None:
            return mejorp, mejord
            
        #reviso si dónde estoy, la distancia es más pequeña para guardarla 
        if mejorp is None or self.distancia(actual.punto, target)< mejord: 
            mejorp= actual.punto
            mejord= self.distancia(actual.punto, target)
            
        #necesito saber sobre qué eje lo voy a comparar 
        c= nivel% self.h
        
        #necesito guardar sobre qué rama voy a bajar 
        rama= "izq"

        #veo qué pasa en mi punto, ¿por dónde bajo? derecha o izquierda
        if target[c]<= actual.punto[c]: 
            mejorp, mejord = self.buscar_min(actual.izq, target, nivel+1, mejorp, mejord)
        else: 
            #si bajo por mi derecha, voy a necesitar cambiar mi ramita
            mejorp, mejord= self.buscar_min(actual.der, target, nivel+1, mejorp, mejord)
            rama= "der"
            
        #necesito ver si vale la pena analizar la otra rama una vez que ya tengo mi candidato   
        if mejord > abs(actual.punto[c]- target[c]): 
            if rama== "izq": 
                mejorp, mejord= self.buscar_min(actual.der, target, nivel+1, mejorp, mejord)
            else: 
                mejorp, mejord= self.buscar_min(actual.izq, target, nivel+1, mejorp, mejord)
        return mejorp, mejord     


    def distancia(self, punto_a, punto_b):
      R = 6371.0  # radio de la Tierra                    
      dlat = math.radians(punto_b[0] - punto_a[0])                            
      dlon = math.radians(punto_b[1] - punto_a[1])                            
      a = (math.sin(dlat/2)**2 + math.cos(math.radians(punto_a[0])) * math.cos(math.radians(punto_b[0])) * math.sin(dlon/2)**2) 
        
      return R * 2 * math.asin(math.sqrt(a))  # retorna km

    def busca_radio(self, actual, target, nivel, radio, aptos): 
        #nota: target es el que le meto, actual es sobre el que estoy iterando 
        if actual is None:
            return aptos 
            
        #reviso si dónde estoy, la distancia es más pequeña para guardarla 
        if self.distancia(actual.punto, target)< radio: 
            aptos.append(actual.punto)
            
        #necesito saber sobre qué eje lo voy a comparar 
        c= nivel% self.h
        
        #necesito guardar sobre qué rama voy a bajar 
        rama= "izq"

        #veo qué pasa en mi punto, ¿por dónde bajo? derecha o izquierda
        if target[c]<= actual.punto[c]: 
            self.busca_radio(actual.izq, target, nivel+1, radio, aptos)
        else: 
            #si bajo por mi derecha, voy a necesitar cambiar mi ramita
            self.busca_radio(actual.der, target, nivel+1, radio, aptos)
            rama= "der"
            
        #necesito ver si vale la pena analizar la otra rama una vez que ya tengo mi candidato   
        if radio > abs(actual.punto[c]- target[c]): 
            if rama== "izq": 
                self.busca_radio(actual.der, target, nivel+1, radio, aptos)
            else: 
                self.busca_radio(actual.izq, target, nivel+1, radio, aptos)
        return aptos

In [5]:
import csv 
def cargar_lugares(archivo="lugares.csv"):
    lugares = {}
    with open(archivo, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            lid = int(row["id"])
            lugares[lid] = {
                "nombre": row["nombre"],
                "tipo": row["tipo"],
                "lat": float(row["latitud"]),
                "lon": float(row["longitud"]),
                "rating": float(row["rating"]),
                "tiempo_visita": int(row["tiempo_visita_min"]),
            }
    return lugares


def cargar_conexiones(archivo="conexiones.csv"):
    aristas = []
    with open(archivo, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            aristas.append((
                int(row["origen"]),
                int(row["destino"]),
                int(row["tiempo_min"]),
                int(row["costo_pesos"]),
            ))
    return aristas


def cargar_horarios(archivo="horarios.csv"):
    horarios = {}
    with open(archivo, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            horarios[int(row["id_lugar"])] = {
                "abre": row["abre"],
                "cierra": row["cierra"],
                "dias_cerrado": row["dias_cerrado"],
            }
    return horarios


def cargar_hoteles(archivo="hoteles.csv"):
    hoteles = {}
    with open(archivo, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            hid = int(row["id"])
            hoteles[hid] = {
                "nombre": row["nombre"],
                "lat": float(row["latitud"]),
                "lon": float(row["longitud"]),
            }
    return hoteles



### Aquí empieza el código

##### La función de lugares_cercanos se enfoca en que tu le metas un punto y la distancia a la redonda y le regresemos al usuario todas las opciones turísticas que puede recorrer.

In [6]:
def lugares_cercanos(punto_actual, radio_permitido, lugares): 
    dicc_puntos= {}
    arbol_puntos= kd_trees(2)
    lista= []
    aptos= []
    lugares_cercanos= {}

    #Sacamos la tupla de puntos de cada lugar para los nodos del kd_tree
    for idx in lugares: 
        dicc_puntos[idx]= lugares[idx]["lat"],lugares[idx]["lon"]

    for val in dicc_puntos.values(): 
        lista.append(val)

    #construimos el kd_tree con los valores
    arbol_puntos.construir_arbol(lista, 0)
    aptos= arbol_puntos.busca_radio(arbol_puntos.raiz, punto_actual, 0, radio_permitido, [])

    # A partir de los puntos aptos, sacamos sus índices para poder sacar los lugares cercanos
    for key, valor in dicc_puntos.items() : 
        if valor in aptos:
            lugares_cercanos[key]= lugares[key]
            
    return lugares_cercanos

In [7]:
def distancia_km(punto_a, punto_b):
      R = 6371.0  # radio de la Tierra                    
      dlat = math.radians(punto_b[0] - punto_a[0])                            
      dlon = math.radians(punto_b[1] - punto_a[1])                            
      a = (math.sin(dlat/2)**2 + math.cos(math.radians(punto_a[0])) * math.cos(math.radians(punto_b[0])) * math.sin(dlon/2)**2) 
        
      return R * 2 * math.asin(math.sqrt(a))  # retorna km

##### La función de grafo se encarga de crear un grafo a partir de la conexiones dadas en el csv

In [8]:
def grafo_original(aristas):
    hijos={}                                                                                              
    for origen, destino, tiempo, costo in aristas:
        if origen not in hijos:                                                                           
            hijos[origen]={}                                                                              
        if destino not in hijos:                                                                          
            hijos[destino]={}                                                                             
        hijos[origen][destino]= tiempo                                                                    
        hijos[destino][origen]= tiempo
                                                                                                            
    conexiones_tiempo= grafos()                                                                           
    conexiones_tiempo.G= hijos

    return conexiones_tiempo 

##### La función de comienzo se encarga de buscar el punto más cercano al hotel y asignarlo como el primer punto a visitar. De esta manera no necesitamos agregar un lugar extra a lugares

In [9]:
def comienzo(opciones, punto_usuario):
    dicc_puntos= {}
    arbol_puntos= kd_trees(2)
    lista= []
    comienzo= 0

    #Sacamos la tupla de puntos de cada lugar para los nodos del kd_tree
    for idx in opciones: 
        dicc_puntos[idx]= opciones[idx]["lat"],opciones[idx]["lon"]

    for val in dicc_puntos.values(): 
        lista.append(val)

    #construimos el kd_tree con los valores
    arbol_puntos.construir_arbol(lista, 0)
    cercano= arbol_puntos.buscar_min(arbol_puntos.raiz, punto_usuario, 0, None, None)
    
    for key, valor in dicc_puntos.items() : 
        if valor == cercano[0]:
            comienzo= key

    return comienzo, cercano[1]

##### Como no podemos asegurar que el camino más corto esté conectado para poder hacer un prim, entonces haremos un truquito. 
##### Haremos conexiones artificiales, es decir, conexiones directas entre el punto de inicio del usuario y los puntos que queremos visitar, pero con el peso mínimo posible. De esta manera, si el camino más corto no está conectado, el algoritmo de prim va a elegir estas conexiones artificiales y nos va a dar el camino más corto de todas formas. Esto también lo haremos con los demás lugares para asegurarnos que el prim hará verdaderamente la conexión más óptima.

In [10]:
def conexiones_artificialesF(inicio_usuario, conexiones_tiempo, opciones):
    pesos= {}
    conexiones_artificiales= {}
    pesos[inicio_usuario], conexiones_artificiales[inicio_usuario]= conexiones_tiempo.dijkstra(inicio_usuario)
    for opcion in opciones: 
        pesos[opcion], conexiones_artificiales[opcion]= conexiones_tiempo.dijkstra(opcion)
    return pesos, conexiones_artificiales

##### Una vez que ya tenemos las conexiones con los pesos que buscamos, vamos a crear un grafo con ellos para poder hacer un prim

In [11]:
def ruta_prim(pesos, opciones, inicio_usuario):
    conexiones_artificiales_grafo= {} 

    for idx, valores in pesos.items(): 
        conexiones_artificiales_grafo[idx]= {} 

        for conexion, valor in valores.items(): 

            if conexion not in conexiones_artificiales_grafo and conexion in opciones.keys():
                conexiones_artificiales_grafo[conexion] = {}

            if valor != float('inf') and valor != 0 and conexion in opciones.keys():
                conexiones_artificiales_grafo[idx][conexion] = valor
    grafo_artificial= grafos() 
    grafo_artificial.G= conexiones_artificiales_grafo 
    conexiones_optimas= grafo_artificial.prim(inicio_usuario)

    return conexiones_optimas

##### Ya que tenemos el camino más óptimo, va a ser necesario que reconstruyamos el camino que pudo haber recortado Dijkstras

In [12]:
def reconstruir_camino(pi, destino):
    camino = []                                                                                       
    actual = destino                                                                                  
    while actual is not None:                                                                         
        camino.append(actual)                                                                         
        actual = pi[actual]                                                                           
    camino.reverse()                                                                                  
    return camino   

In [13]:
def camino_reconstruido(conexiones_optimas, conexiones_artificiales, lugares): 
    for destino, origen in conexiones_optimas.items():                                                    
      if origen is not None:                                                                            
          pi_real = conexiones_artificiales[origen]
          #Este es el camino real entre el padre y el hijo, que es lo que nos da el Dijkstra desde el padre                                                     
          camino = reconstruir_camino(pi_real, destino) 

          #Esto es para el print, lo podemos quitar después    
          for n in camino:                                             
            nombres = [lugares[n]["nombre"]]    
          print(lugares[origen]["nombre"], "→", lugares[destino]["nombre"], "(", pesos[origen][destino], "min )")
          #----------------------------------------------------
    return camino

In [14]:
def orden_visita_ponderado(pi, inicio, horarios, lugares):
     # NOTA: el orden de la ponderación es: 
     # cierre, rating, distancia, tiempo de visita.
      #Para crear la lista de hijos con el que hacemos el DFS                                         
      hijos_prim = {}                                                                                      
      for nodo in pi:                                                                                     
          hijos_prim[nodo] = []                                                                            
      for nodo, padre in pi.items():                                                                      
          if padre is not None:                                                                           
              hijos_prim[padre].append(nodo)                                                               
                                                                                                          
      orden = []                                                                                          
      pila = [inicio]                                                                                     
      while pila:                                                                                         
          actual = pila.pop()                                                                             
          orden.append(actual)                                                                            
                                                                                                          
          hijos = hijos_prim[actual]

          #Si no tienes hijos, salta todo esto que es para ponderar los hijos                                                                       
          if len(hijos) == 0:                                                                             
              continue                                                                                    
                  
          puntajes = {}                                                                                   
          for hijo in hijos:
              #Todo esta parte es para asignarle ponderación a cada hijo y así poder ordenarlos antes de meterlos a 
              # la pila, para que el que tenga más puntaje, se meta después y así se saque antes
              puntaje = 0                                                                                 

              #Ponderamos para que los que cierran más tarde, tengan menos puntaje                                                                                          
              if hijo in horarios:  
                  # Estandarizamos cómo vamos a medir las horas                                                                       
                  cierre = int(horarios[hijo]["cierra"].replace(":", ""))                                 
              else: # No hay horario de cierre                                                                                      
                  cierre = 2359
              puntaje = puntaje +  (2400 - cierre)                               

              # Ponderamos para que los de mejor rating tengan más puntaje                                                                                            
              puntaje = puntaje + (lugares[hijo]["rating"] * 100)               

              # Ponderamos la distancia entre el lugar actual y el hijo
              # Usamos la función de distancia del kd_tree                                                                                              
              dist = kd_trees(2)                                                                          
              punto_a = (lugares[actual]["lat"], lugares[actual]["lon"])
              punto_b = (lugares[hijo]["lat"], lugares[hijo]["lon"])                                      
              km = dist.distancia(punto_a, punto_b)
              #Restamos a 100 para que a menor distancia, mayor puntaje, y así se ordene primero                                                       
              puntaje = puntaje + (100 - km)                                 

              # Ponderamos para el tiempo visita: a mayor tiempo de visita, mayor puntaje 
              # La idea es visitar primero los que requieren más tiempo de visita 
              puntaje = puntaje + lugares[hijo]["tiempo_visita"]         
                                                                                                          
              puntajes[hijo] = puntaje                                                                    
                  
          hijos_con_puntaje = []                                                                          
          for hijo in hijos:
              hijos_con_puntaje.append((puntajes[hijo], hijo))  

          # Acomodar el puntaje para que el de mayor puntaje entre después                                               
          hijos_con_puntaje.sort() 
          #Para meterlos a la cola, ignoramos el puntaje aquí                                                                    
          for puntaje_h, hijo in hijos_con_puntaje:                                                       
              pila.append(hijo)                                                                           
                                                                                                          
      return orden 


In [15]:
def orden_visita_no_real_sin_ponderación(pi, inicio):                                                                         
    """Convierte el prim en un orden secuencial de visita usando DFS"""
    # Recordatorio: DFS es primero hasta lo más profundo, luego vuelve para revisar otras ramas.                           
    hijos_prim = {}                                                                                    
    for nodo in pi:                                                                                   
        hijos_prim[nodo] = []                                                                          
    for nodo, padre in pi.items():                                                                    
        if padre is not None:                                                                         
            hijos_prim[padre].append(nodo)                                                             
                                                                                                    
    # DFS para sacar el orden
    orden = []                                                                                        
    pila = [inicio]                                                                                   
    while pila:
        #Usamos la pila para claramente acabar una rama antes de pasar a la siguiente                                                                                       
        actual = pila.pop()                                                                           
        orden.append(actual)                                                                          
        for hijo in hijos_prim[actual]:                                                                
            pila.append(hijo)                                                                         
                                                                                                    
    return orden          

#### Cargas de archivos necesarios y alimentar el Trie

In [16]:
lugares= cargar_lugares()
hoteles= cargar_hoteles()
aristas=cargar_conexiones()
horarios= cargar_horarios()

In [17]:
autocompletado= Trie()
#Para insertar los nombres de los lugares en el trie para el autocompletado
for idx,info in lugares.items(): 
    autocompletado.inserta(info["nombre"])
# Para insertar los nombres de los hoteles en el trie para el autocompletado
for idx,info in hoteles.items():
    autocompletado.inserta(info["nombre"])

In [18]:
# Planeación interacción con el usuario: Se le pregunta en que hotel se está hospedando (se usa autocompletar con el trie y se le dan opciones numéricas para que elija a cual se refiere)
# Luego, se le pregunta si desea elegir los lugares a visitar manualmente o que le mostremos los que estén a menos de n (input) kilometros a la redonda, si desea elegir manualmente 
# será autocompletar con Trie. Si le devolvemos los que tiene cerca entonces elige separado por comas cuales quiere visitar interaccion que ya estña en la celda de abajo

In [19]:
def interaccion_usuario(hoteles, lugares, autocompletado):
      # --- PASO 1: Elegir hotel ---         
      print("¿En qué hotel te hospedas? Escribe las primeras letras:")                                    
      prefijo_hotel = input("  → ")                                                                       
      autocompletado.autocompletado(prefijo_hotel)                                                        
                                                                                                          
      # mostrar opciones numéricas de hoteles que coinciden                                               
      opciones_hotel = {}                                                                                 
      for hid, info in hoteles.items():                                                                   
          if info["nombre"].lower().startswith(prefijo_hotel.lower()):                                    
              opciones_hotel[hid] = info                                                                  
              print(f"  {hid}. {info['nombre']}")                                                         
                                                                                                          
      hotel_elegido = int(input("Escribe el número del hotel: "))                                         
      punto_usuario = hoteles[hotel_elegido]["lat"], hoteles[hotel_elegido]["lon"]                        
      print(f"  OK: {hoteles[hotel_elegido]['nombre']}\n")                                                
                                                                                                          
      # --- PASO 2: Elegir cómo buscar lugares ---                                                        
      print("¿Cómo deseas elegir los lugares a visitar?")                                                 
      print("  1. Ver lugares cercanos a mi hotel")                                                       
      print("  2. Buscar manualmente por nombre")                                                         
      modo = input("  → ")                                                                                
                                                                                                          
      opciones_final = {}                                                                                 
                                                                                                          
      if modo == "1":                                                                                     
          radio = float(input("¿A cuántos km a la redonda? → "))
          opciones = lugares_cercanos(punto_usuario, radio, lugares)                                      
          print("\nLugares cercanos:")                                                                    
          for lid, info in opciones.items():                                                              
              print(f"  {lid}. {info['nombre']} ({info['tipo']}, rating {info['rating']})")               
                                                                                                          
          elegidos_str = input("\n¿Cuáles quieres visitar? (números separados por coma): ")               
          partes = elegidos_str.split(",")                                                                
          for parte in partes:                                                                            
              lid = int(parte.strip())                                                                    
              if lid in opciones:                                                                         
                  opciones_final[lid] = opciones[lid]                                                     
              else:                                                                                       
                  print(f"  {lid} no está en las opciones cercanas, se omite")                            
                                                                                                          
      else:                                                                                               
          print("\nEscribe los nombres (o prefijos) de los lugares que quieres visitar.")                 
          print("Escribe 'listo' cuando termines.\n")                                                     
          while True:                                                                                     
              prefijo = input("  Lugar → ")                                                               
              if prefijo.lower() == "listo":                                                              
                  break                                                                                   
                                                                                                          
              # buscar coincidencias con el prefijo                                                       
              coincidencias = {}                                                                          
              for lid, info in lugares.items():                                                           
                  if info["nombre"].lower().startswith(prefijo.lower()):                                  
                      coincidencias[lid] = info                                                           
                      print(f"    {lid}. {info['nombre']}")                                               
                                                                                                          
              if len(coincidencias) == 0:                                                                 
                  print("    No se encontró nada con ese prefijo")                                        
                  continue                                                                                
                                                                                                          
              elegido = int(input("  ¿Cuál? (número) → "))                                                
              if elegido in coincidencias:                                                                
                  opciones_final[elegido] = lugares[elegido]                                              
                  print(f"    Agregado: {lugares[elegido]['nombre']}\n")                                  
              else:                                                                                       
                  print("    Número no válido")                                                          
                                                                                                          
      print(f"\nLugares seleccionados:")                                                                  
      for lid, info in opciones_final.items():                                                            
          print(f"  - {info['nombre']}")                                                                  
                                                                                                          
      return opciones_final, punto_usuario            

In [ ]:
conexiones_tiempo = grafo_original(aristas)
opciones_final, punto_usuario = interaccion_usuario(hoteles, lugares, autocompletado)
inicio_usuario, distancia_hotel = comienzo(opciones_final, punto_usuario)                               
inicio_usuario

¿En qué hotel te hospedas? Escribe las primeras letras:
Hotel Zócalo Central
Hotel Hilton Reforma
Hotel Condesa DF
Hotel Camino Real Polanco
Hotel Casa González Roma
Hotel Four Seasons
Hotel NH Centro Histórico
Hotel Xochimilco Resort
  1. Hotel Zócalo Central
  2. Hotel Hilton Reforma
  3. Hotel Condesa DF
  4. Hotel Four Seasons
  5. Hotel NH Centro Histórico
  6. Hotel Camino Real Polanco
  7. Hotel Casa González Roma
  8. Hotel Xochimilco Resort
  OK: Hotel Four Seasons

¿Cómo deseas elegir los lugares a visitar?
  1. Ver lugares cercanos a mi hotel
  2. Buscar manualmente por nombre


In [102]:
print(opciones_final.keys())

dict_keys([3, 6, 10])


In [103]:
pesos, conexiones_artificiales= conexiones_artificialesF(inicio_usuario, conexiones_tiempo, opciones_final)
conexiones_artificiales

{10: {1: 10, 2: 10, 8: 1, 10: None, 7: 2, 3: 2, 9: 8, 5: 9, 4: 5, 6: 10},
 3: {1: 2, 2: 3, 8: 9, 10: 2, 7: 3, 3: None, 9: 3, 5: 3, 4: 5, 6: 10},
 6: {1: 10, 2: 10, 8: 1, 10: 6, 7: 2, 3: 2, 9: 5, 5: 4, 4: 6, 6: None}}

In [104]:
conexiones_optimas= ruta_prim(pesos, opciones_final, inicio_usuario)
conexiones_optimas

{10: None, 3: 10, 6: 10}

In [105]:
camino= camino_reconstruido(conexiones_optimas, conexiones_artificiales, lugares)

Tlatelolco → Chapultepec ( 42 min )
Tlatelolco → Teotihuacán ( 95 min )


In [106]:
orden = orden_visita_ponderado(conexiones_optimas, inicio_usuario, horarios, lugares)                                              
  # orden = [1, 10, 2, 8, 9]                                                                            


In [107]:
def hora_max_salida(orden, pesos, horarios, lugares, dist_hotel_km, velocidad_kmh=5):                   
      # tiempo caminando del hotel al primer lugar                              
      tiempo_hotel_inicio = (dist_hotel_km / velocidad_kmh) * 60                                          
                                                                                                          
      # acumulamos el tiempo desde que sales del hotel                                                    
      tiempos_llegada = {}                                                                                
      acumulado = tiempo_hotel_inicio                                                                     
                                                                                                          
      # llegada al primer lugar                                                                            
      tiempos_llegada[orden[0]] = acumulado                                                               
                                                                                                          
      for i in range(len(orden) - 1):                                                                     
          origen = orden[i]                                                                               
          destino = orden[i + 1]                                                                          
                                                                                                          
          # sumamos el tiempo de visita del lugar actual                                                    
          acumulado = acumulado + lugares[origen]["tiempo_visita"]                                        
                                                                                                          
          # sumamos el tiempo de traslado al siguiente                                                      
          acumulado = acumulado + pesos[origen][destino]                                                  
                                                                                                          
          # guardamos a qué minuto llegas al siguiente                                                      
          tiempos_llegada[destino] = acumulado                                                            
                                                                                                          
      # ahora vemos la restricción de cada lugar con horario                                              
      hora_limite = None                                                                                  
      lugar_limitante = None                                                                              
                                                                                                          
      for lugar_id, minutos_llegada in tiempos_llegada.items():                                           
          if lugar_id not in horarios:                                                                    
              continue                                                                                    
                                                                                                          
          # convertir hora de cierre a minutos desde medianoche                                           
          cierre_str = horarios[lugar_id]["cierra"]                                                       
          partes = cierre_str.split(":")                                                                  
          cierre_min = int(partes[0]) * 60 + int(partes[1])                                               
                                                                                                          
          # hora máxima de salida para llegar a este lugar antes de que cierre                            
          max_salida = cierre_min - minutos_llegada - lugares[lugar_id]["tiempo_visita"]                                                               
                                                                                                          
          if hora_limite is None or max_salida < hora_limite:                                             
              hora_limite = max_salida                                                                    
              lugar_limitante = lugar_id                                                                  
                                                                                                          
      if hora_limite is None:                                                                             
          print("Ningún lugar tiene restricción de horario, puedes salir a cualquier hora")               
          return None                                                                                     
                  
      # convertir minutos a formato hora                                                                  
      horas = int(hora_limite) // 60
      minutos = int(hora_limite) % 60    
      print("Debes salir a más tardar a las", f"{horas}:{minutos:02d}.", "Restricción por", lugares[lugar_limitante]['nombre'], "(cierra a las", f"{horarios[lugar_limitante]['cierra']})")                                                                
                                                                                                         
      return hora_limite

In [ ]:
def hora_max_salida(orden, pesos, horarios, lugares, dist_hotel_km, velocidad_kmh=5):

    # tiempo caminando del hotel al primer lugar                                
    tiempo_hotel_inicio = (dist_hotel_km / velocidad_kmh) * 60               
                                                                                                                                                                                                                                                                                                                                         
    # acumulamos el tiempo desde que el usuario sale del hotel                                                    
    tiempos_llegada = {}                                                                                
    acumulado = tiempo_hotel_inicio     

    # llegada al primer lugar                                                                                                                                            
    tiempos_llegada[orden[0]] = acumulado                                                               
                                                                                                        
    for i in range(len(orden) - 1):                                                                     
        origen = orden[i]                                                                               
        destino = orden[i + 1]      
        # sumamos el tiempo de visita del lugar actual                                                    
        acumulado = acumulado + lugares[origen]["tiempo_visita"]                                        
                                                                                                        
        # sumamos el tiempo de traslado al siguiente                                                      
        acumulado = acumulado + pesos[origen][destino]                                                  
                                                                                                        
        # guardamos a qué minuto el usuario al siguiente                                                      
        tiempos_llegada[destino] = acumulado                                                            
                                                                                                                                                                                  
    # Encontrar hora límite de salida viendo la restricción de cada lugar con horario                                                                 
    hora_limite = None                                                                                  
    lugar_limitante = None                                                                              
                                                                                                        
    for lugar_id, minutos_llegada in tiempos_llegada.items():                                           
        if lugar_id not in horarios:                                                                    
            continue                                                                                    
        cierre_str = horarios[lugar_id]["cierra"]
        # convertir hora de cierre a minutos desde medianoche                                           
        partes = cierre_str.split(":")                                                                  
        cierre_min = int(partes[0]) * 60 + int(partes[1])  
        # hora máxima de salida para llegar a este lugar antes de que cierre                            
        max_salida = cierre_min - minutos_llegada - lugares[lugar_id]["tiempo_visita"]                  
                                                                                                        
        if hora_limite is None or max_salida < hora_limite:                                             
            hora_limite = max_salida                                                                    
            lugar_limitante = lugar_id                                                                  
                                                                                                        
    if hora_limite is None:                                                                             
        print("Ningún lugar tiene restricción de horario, puedes salir a cualquier hora")               
        return None                                                                                     
                                                                                                        
    # Segunda pasada: con la hora de salida, ver a qué hora real llega el usuario a cada lugar                    
    # y cuáles están cerrados (no abiertos aún, ya cerraron, no alcanza a visitarlo)                 
    no_visitables = []                                                                                  
    acumulado = tiempo_hotel_inicio                                                                     
    tiempos_llegada_real = {}                                                                           
    tiempos_llegada_real[orden[0]] = acumulado                                                          
                                                                                                        
    for i in range(len(orden) - 1):                                                                     
        origen = orden[i]                                                                               
        destino = orden[i + 1]                                                                          
                                                                                                        
        # solo sumamos tiempo de visita si el lugar está abierto                                     
        hora_llegada_origen = hora_limite + tiempos_llegada_real[origen]                                
        abierto_origen = True                                                                           
                                                                                                        
        if origen in horarios:                                                                          
            abre_str = horarios[origen]["abre"]                                                         
            cierre_str = horarios[origen]["cierra"]                                                     
            partes_a = abre_str.split(":")                                                              
            partes_c = cierre_str.split(":")                                                            
            abre_min = int(partes_a[0]) * 60 + int(partes_a[1])                                         
            cierre_min = int(partes_c[0]) * 60 + int(partes_c[1])                                       
                                                                                                        
            # cerrado si el usuario llega antes de que abra o si no alcanza a terminar la visita                       
            if hora_llegada_origen < abre_min:                                                          
                abierto_origen = False                                                                  
            if hora_llegada_origen + lugares[origen]["tiempo_visita"] > cierre_min:                     
                abierto_origen = False                                                                  
                                                                                                        
        if abierto_origen:                                                                              
            acumulado = acumulado + lugares[origen]["tiempo_visita"]                                    
        else:                                                                                           
            if origen not in no_visitables:                                                             
                no_visitables.append(origen)                                                            
                                                                                                        
        acumulado = acumulado + pesos[origen][destino]                                                  
        tiempos_llegada_real[destino] = acumulado                                                       
                                                                                                        
    # revisar el ultimo lugar del orden                                                          
    ultimo = orden[len(orden) - 1]                                                                      
    hora_llegada_ultimo = hora_limite + tiempos_llegada_real[ultimo]                                    
    if ultimo in horarios:                                                                              
        abre_str = horarios[ultimo]["abre"]                                                             
        cierre_str = horarios[ultimo]["cierra"]                                                         
        partes_a = abre_str.split(":")                                                                  
        partes_c = cierre_str.split(":")                                                                
        abre_min = int(partes_a[0]) * 60 + int(partes_a[1])                                             
        cierre_min = int(partes_c[0]) * 60 + int(partes_c[1])                                           
        if hora_llegada_ultimo < abre_min:                                                              
            no_visitables.append(ultimo)                                                                
        elif hora_llegada_ultimo + lugares[ultimo]["tiempo_visita"] > cierre_min:                       
            no_visitables.append(ultimo)                                                                
                                                                                                        
    # Imprimir resultado                                                                                
    horas = int(hora_limite) // 60                                                                      
    minutos = int(hora_limite) % 60     
    print()                                                                                             
    print("Debes salir a más tardar a las", f"{horas}:{minutos:02d}.")                                  
    print("  Restricción por", lugares[lugar_limitante]["nombre"], "(cierra a las",f"{horarios[lugar_limitante]['cierra']})")                                                              
    print()                                                                                             
                                                                                                        
    if len(no_visitables) > 0:                                                                          
        print("Según los tiempos de visita que nuestros usuarios reportan por cada atracción,")         
        print("no podemos asegurarte que tengas oportunidad de entrar a todos los lugares que deseas.") 
        print("A continuación te mostramos cuáles probablemente no puedas entrar y solo puedas ver por fuera:")                                                                                                
        for lid in no_visitables:                                                                       
            motivo = ""                                                                                 
            if lid in horarios:                                                                         
                hora_ll = hora_limite + tiempos_llegada_real[lid]                                       
                h_ll = int(hora_ll) // 60                                                               
                m_ll = int(hora_ll) % 60                                                                
                motivo = f"(llegas a las {h_ll}:{m_ll:02d}, abre {horarios[lid]['abre']}, cierra {horarios[lid]['cierra']})"                                                                             
            print(f"  - {lugares[lid]['nombre']} {motivo}")                                             
        print()                                                                                         
        print("Te recomendamos reconsiderar tus lugares a visitar.")                                    
                                                                                                        
    return hora_limite, no_visitables                                                                                                                                  
                                               

In [109]:
                                                                                                        
print("=== RECORRIDO EN ORDEN ===\n")                                                                 
tiempo_hotel_min = (distancia_hotel / 5) * 60                                                           
print(f"Hotel → {lugares[orden[0]]['nombre']} ({tiempo_hotel_min:.0f} min caminando,{distancia_hotel:.2f} km)")                                                                             
total_tiempo = int(tiempo_hotel_min)                                                                     
for i in range(len(orden) - 1):                                                                       
    origen = orden[i]                                                                                 
    destino = orden[i + 1]                                                                            
                                                                                                    
    # Ruta real entre consecutivos                                                                    
    pi_real = conexiones_artificiales[origen]                                                         
    camino = reconstruir_camino(pi_real, destino)                                                     
    tiempo = pesos[origen][destino]                                                                   
    total_tiempo += tiempo                                                                            
                                                                                                    
    nombres = [lugares[n]["nombre"] for n in camino]        
    print(lugares[origen]["nombre"], "→", lugares[destino]["nombre"], "(", tiempo, "min )")
    print("    Ruta: ", " → ".join(nombres))                                          

print("Total traslado: ", total_tiempo, "min")       
                                                                                            
hora_limite= hora_max_salida(orden, pesos, horarios, lugares, distancia_hotel)

=== RECORRIDO EN ORDEN ===

Hotel → Tlatelolco (26 min caminando,2.20 km)
Tlatelolco → Teotihuacán ( 95 min )
    Ruta:  Tlatelolco → Teotihuacán
Teotihuacán → Chapultepec ( 137 min )
    Ruta:  Teotihuacán → Tlatelolco → Bellas Artes → Chapultepec
Total traslado:  258 min

Debes salir a más tardar a las 5:26.
  Restricción por Chapultepec (cierra a las 18:00)

Según los tiempos de visita que nuestros usuarios reportan por cada atracción,
no podemos asegurarte que tengas oportunidad de entrar a todos los lugares que deseas.
A continuación te mostramos cuáles probablemente no puedas entrar y solo puedas ver por fuera:
  - Teotihuacán (llegas a las 8:43, abre 09:00, cierra 17:00)

Te recomendamos reconsiderar tus lugares a visitar.
